In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Set visualization style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 11

# Constants
FHCF_CAP = 17e9  # $17B statewide cap

## 1. Load ERA5 Baseline Data

Load the ERA5 baseline Monte Carlo results (10,000 simulated years).

In [ ]:
# Load ERA5 baseline iterations
baseline_path = '../results/mc_runs/emanuel_era5_baseline_20251207_072016'
#baseline_path = '../results/mc_runs/emanuel_era5_baseline_20260204_140957'

df_era5 = pd.read_csv(f'{baseline_path}/iterations.csv')

print(f"Loaded ERA5 simulation with {len(df_era5):,} iterations")
print(f"\nColumns available: {list(df_era5.columns)[:20]}")
print(f"\nScenario types: {df_era5['scenario'].unique()[:10]}")

### 1.1 Data Quality Check

Check for error years and basic statistics.

In [ ]:
# Count error vs successful years
error_years = (df_era5['scenario'] == 'error').sum()
successful_years = len(df_era5) - error_years

print("=" * 80)
print("DATA QUALITY SUMMARY")
print("=" * 80)
print(f"Total iterations: {len(df_era5):,}")
print(f"Successful years: {successful_years:,} ({successful_years/len(df_era5)*100:.1f}%)")
print(f"Error years: {error_years:,} ({error_years/len(df_era5)*100:.1f}%)")
print("=" * 80)

# Filter to successful years only for analysis
df_valid = df_era5[df_era5['scenario'] != 'error'].copy()

print(f"\nAnalysis dataset: {len(df_valid):,} valid years")

### 1.2 Descriptive Statistics

Summary statistics for key metrics across all simulated years.

In [ ]:
# Loss statistics (in billions)
print("\n" + "=" * 80)
print("LOSS STATISTICS (Billion USD)")
print("=" * 80)

print(f"\nTotal damage:")
print(f"  Mean: ${df_valid['total_damage_usd'].mean()/1e9:.2f}B")
print(f"  Median: ${df_valid['total_damage_usd'].median()/1e9:.2f}B")
print(f"  90th percentile: ${df_valid['total_damage_usd'].quantile(0.9)/1e9:.2f}B")
print(f"  99th percentile: ${df_valid['total_damage_usd'].quantile(0.99)/1e9:.2f}B")
print(f"  Max: ${df_valid['total_damage_usd'].max()/1e9:.2f}B")

print(f"\nWind losses:")
print(f"  Mean: ${df_valid['wind_total_usd'].mean()/1e9:.2f}B")
print(f"  90th percentile: ${df_valid['wind_total_usd'].quantile(0.9)/1e9:.2f}B")

print(f"\nFlood losses:")
print(f"  Mean: ${df_valid['water_total_usd'].mean()/1e9:.2f}B")
print(f"  90th percentile: ${df_valid['water_total_usd'].quantile(0.9)/1e9:.2f}B")

print(f"\nCitizens wind:")
print(f"  Mean: ${df_valid['wind_insured_citizens_usd'].mean()/1e9:.2f}B")
print(f"  90th percentile: ${df_valid['wind_insured_citizens_usd'].quantile(0.9)/1e9:.2f}B")

# Default statistics
print("\n" + "=" * 80)
print("PRIVATE MARKET DEFAULT STATISTICS")
print("=" * 80)
print(f"Years with ANY defaults: {(df_valid['defaults_post'] > 0).sum():,} ({(df_valid['defaults_post'] > 0).sum()/len(df_valid)*100:.1f}%)")
print(f"Years with >5 defaults: {(df_valid['defaults_post'] > 5).sum():,} ({(df_valid['defaults_post'] > 5).sum()/len(df_valid)*100:.1f}%)")
print(f"Years with >10 defaults: {(df_valid['defaults_post'] > 10).sum():,} ({(df_valid['defaults_post'] > 10).sum()/len(df_valid)*100:.1f}%)")

defaults_present = df_valid[df_valid['defaults_post'] > 0]
if len(defaults_present) > 0:
    print(f"\nDefaults when present:")
    print(f"  Mean: {defaults_present['defaults_post'].mean():.1f}")
    print(f"  Median: {defaults_present['defaults_post'].median():.1f}")
    print(f"  Max: {defaults_present['defaults_post'].max():.0f}")

# Public burden statistics
print("\n" + "=" * 80)
print("PUBLIC BURDEN STATISTICS (Billion USD)")
print("=" * 80)

fhcf_shortfall = df_valid['fhcf_shortfall_usd'] / 1e9
citizens_deficit = df_valid['citizens_residual_deficit_usd'] / 1e9
nfip_borrowed = df_valid['nfip_borrowed_usd'] / 1e9
total_public = fhcf_shortfall + citizens_deficit + nfip_borrowed

print(f"Total public burden:")
print(f"  Years with burden > 0: {(total_public > 0).sum():,} ({(total_public > 0).sum()/len(df_valid)*100:.1f}%)")
print(f"  Mean (all years): ${total_public.mean():.2f}B")
print(f"  Mean (when > 0): ${total_public[total_public > 0].mean():.2f}B")
print(f"  Median (when > 0): ${total_public[total_public > 0].median():.2f}B")
print(f"  90th percentile: ${total_public.quantile(0.9):.2f}B")
print(f"  99th percentile: ${total_public.quantile(0.99):.2f}B")
print(f"  Max: ${total_public.max():.2f}B")

print(f"\nFHCF shortfall:")
print(f"  Years with shortfall: {(fhcf_shortfall > 0).sum():,} ({(fhcf_shortfall > 0).sum()/len(df_valid)*100:.1f}%)")
if (fhcf_shortfall > 0).sum() > 0:
    print(f"  Mean (when > 0): ${fhcf_shortfall[fhcf_shortfall > 0].mean():.2f}B")

print(f"\nCitizens deficit:")
print(f"  Years with deficit: {(citizens_deficit > 0).sum():,} ({(citizens_deficit > 0).sum()/len(df_valid)*100:.1f}%)")
if (citizens_deficit > 0).sum() > 0:
    print(f"  Mean (when > 0): ${citizens_deficit[citizens_deficit > 0].mean():.2f}B")

print(f"\nNFIP borrowing:")
print(f"  Years with borrowing: {(nfip_borrowed > 0).sum():,} ({(nfip_borrowed > 0).sum()/len(df_valid)*100:.1f}%)")
if (nfip_borrowed > 0).sum() > 0:
    print(f"  Mean (when > 0): ${nfip_borrowed[nfip_borrowed > 0].mean():.2f}B")

print("\n" + "=" * 80)

## 2. Probability Analysis: Key Risk Thresholds

Calculate annual exceedance probabilities for critical thresholds across different risk categories:
- **Private Market**: Insurance company defaults
- **Institutional Stress**: FHCF, Citizens, NFIP capacity exceedance
- **System-Wide Failures**: Multiple simultaneous failures

In [ ]:
# Calculate annual exceedance probabilities for key thresholds
print("\n" + "="*80)
print("ANNUAL EXCEEDANCE PROBABILITIES - ERA5 BASELINE")
print("="*80)

# Bootstrap function to compute uncertainty bounds
def compute_uncertainty_bounds(condition_array, n_bootstrap=1000, percentiles=[10, 90]):
    """
    Compute percentile-based uncertainty bounds for binary outcomes using bootstrap.
    
    Parameters:
    -----------
    condition_array : array-like of bool
        Boolean array indicating whether condition is met for each year
    n_bootstrap : int
        Number of bootstrap samples
    percentiles : list
        Percentiles to compute (default [10, 90] for p10-p90 range)
    
    Returns:
    --------
    dict : {'mean': float, 'p10': float, 'p90': float} in percentage
    """
    n_years = len(condition_array)
    bootstrap_probs = []
    
    # Bootstrap resampling
    for _ in range(n_bootstrap):
        sample = np.random.choice(condition_array, size=n_years, replace=True)
        bootstrap_probs.append(sample.mean() * 100)
    
    bootstrap_probs = np.array(bootstrap_probs)
    
    return {
        'mean': condition_array.mean() * 100,
        'p10': np.percentile(bootstrap_probs, percentiles[0]),
        'p90': np.percentile(bootstrap_probs, percentiles[1])
    }

# Calculate metrics directly from the 10,000 stochastic years with uncertainty bounds
metric_results = {}
metric_results_p10 = {}
metric_results_p90 = {}

# Private market metrics
defaults_10 = (df_valid['defaults_post'] > 10)
unc = compute_uncertainty_bounds(defaults_10)
metric_results['Defaults > 10'] = unc['mean']
metric_results_p10['Defaults > 10'] = unc['p10']
metric_results_p90['Defaults > 10'] = unc['p90']

deficit_1b = (df_valid['largest_entity_deficit_usd'] > 1e9)
unc = compute_uncertainty_bounds(deficit_1b)
metric_results['Single Deficit > $1B'] = unc['mean']
metric_results_p10['Single Deficit > $1B'] = unc['p10']
metric_results_p90['Single Deficit > $1B'] = unc['p90']

# Institutional stress metrics
fhcf_cap_exceeded = (df_valid['fhcf_recovery_private_usd'] + df_valid['fhcf_recovery_citizens_usd']) / FHCF_CAP > 1.0
unc = compute_uncertainty_bounds(fhcf_cap_exceeded)
metric_results['FHCF > 100% Cap'] = unc['mean']
metric_results_p10['FHCF > 100% Cap'] = unc['p10']
metric_results_p90['FHCF > 100% Cap'] = unc['p90']

citizens_capacity_exceeded = df_valid['citizens_residual_deficit_usd'] / (df_valid['citizens_tier1_capacity_usd'] + df_valid['citizens_tier2_capacity_usd']) > 1.0
unc = compute_uncertainty_bounds(citizens_capacity_exceeded)
metric_results['Citizens > 100% Capacity'] = unc['mean']
metric_results_p10['Citizens > 100% Capacity'] = unc['p10']
metric_results_p90['Citizens > 100% Capacity'] = unc['p90']

figa_capacity_exceeded = df_valid['figa_residual_deficit_usd'] > 0
unc = compute_uncertainty_bounds(figa_capacity_exceeded)
metric_results['FIGA > 100% Capacity'] = unc['mean']
metric_results_p10['FIGA > 100% Capacity'] = unc['p10']
metric_results_p90['FIGA > 100% Capacity'] = unc['p90']

nfip_stressed = df_valid['nfip_claims_paid_usd'] / df_valid['nfip_fl_premium_base_usd'] > 2.0
unc = compute_uncertainty_bounds(nfip_stressed)
metric_results['NFIP > 200% Annual Premium'] = unc['mean']
metric_results_p10['NFIP > 200% Annual Premium'] = unc['p10']
metric_results_p90['NFIP > 200% Annual Premium'] = unc['p90']

# Update public_stressed to include FIGA (now 4 institutions)
public_stressed = (
    fhcf_cap_exceeded.astype(int) +
    figa_capacity_exceeded.astype(int) +
    citizens_capacity_exceeded.astype(int) +
    nfip_stressed.astype(int)
)

# Total public burden
total_public_burden = (
    df_valid['fhcf_shortfall_usd'] + 
    df_valid['figa_residual_deficit_usd'] + 
    df_valid['citizens_residual_deficit_usd'] + 
    df_valid['nfip_borrowed_usd']
)

# Florida GDP ~$1.7 trillion (2024/25); public burden > 1% of GDP is significant
FL_GDP = 1.7e12  
pub_burden_1pct = (total_public_burden > 0.01 * FL_GDP)
unc = compute_uncertainty_bounds(pub_burden_1pct)
metric_results['Public Burden > 1% FL GDP'] = unc['mean']
metric_results_p10['Public Burden > 1% FL GDP'] = unc['p10']
metric_results_p90['Public Burden > 1% FL GDP'] = unc['p90']

pub_burden_10pct = (total_public_burden > 0.1 * FL_GDP)
unc = compute_uncertainty_bounds(pub_burden_10pct)
metric_results['Public Burden > 10% FL GDP'] = unc['mean']
metric_results_p10['Public Burden > 10% FL GDP'] = unc['p10']
metric_results_p90['Public Burden > 10% FL GDP'] = unc['p90']

# Create DataFrame for visualization
metric_names = [
    'Defaults > 10',
    'Single Deficit > $1B',
    'FHCF > 100% Cap',
    'FIGA > 100% Capacity',
    'Citizens > 100% Capacity',
    'NFIP > 200% Annual Premium',
    'Public Burden > 1% FL GDP',
    'Public Burden > 10% FL GDP'
]

df_era5_metrics = pd.DataFrame({
    'Metric': metric_names,
    'Annual Probability (%)': [metric_results[m] for m in metric_names],
    'P10': [metric_results_p10[m] for m in metric_names],
    'P90': [metric_results_p90[m] for m in metric_names]
})

print("\n" + df_era5_metrics.to_string(index=False))
print("\n" + "="*80)


### 2.1 Visualization: Probabilistic Risk Profile

Visual representation of annual exceedance probabilities for ERA5 baseline.

In [ ]:
from matplotlib.patches import Patch

fig, ax = plt.subplots(figsize=(5, 4))

# Reverse order for display (bottom to top)
df_reversed = df_era5_metrics[::-1].reset_index(drop=True)
values = df_reversed['Annual Probability (%)'].values

# Get uncertainty bounds
p10_vals = df_reversed['P10'].values
p90_vals = df_reversed['P90'].values

# Calculate error bars (asymmetric)
lower_errors = values - p10_vals
upper_errors = p90_vals - values

# Single color for all bars
colors = '#6B7280'

bars = ax.barh(df_reversed['Metric'], values, height=0.5, 
               xerr=[lower_errors, upper_errors],
               color=colors, edgecolor='black', linewidth=0.5,
               error_kw={'linewidth': 1.2, 'ecolor': 'black', 'capsize': 2, 'capthick': 1.2})

# Add category separators
ax.axhline(y=1.5, color='black', linewidth=1.5, xmin=-1.5, xmax=1.0, clip_on=False)
ax.axhline(y=5.5, color='black', linewidth=1.5, xmin=-1.5, xmax=1.0, clip_on=False)

ax.set_xlabel('Annual Probability (%)', fontsize=11, fontweight='normal')
ax.set_xlim(0, max(p90_vals) * 1.15)

# Remove grid
ax.grid(False)

# Style axes
ax.spines['left'].set_color('black')
ax.spines['bottom'].set_color('black')
ax.spines['left'].set_linewidth(1)
ax.spines['bottom'].set_linewidth(1)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.tick_params(axis='both', colors='black')
ax.tick_params(axis='y', labelsize=9)

# Add category labels
fig.text(0.00, 0.85, 'PRIVATE\nMARKET', fontsize=8, fontweight='normal', 
         va='center', ha='center', color='k', rotation=90)
fig.text(0.00, 0.57, 'INSTITUTIONAL\nSTRESS', fontsize=8, fontweight='normal', 
         va='center', ha='center', color='k', rotation=90)
fig.text(0.00, 0.25, 'AGG. FISCAL\nIMPACT', fontsize=8, fontweight='normal', 
         va='center', ha='center', color='k', rotation=90)

plt.tight_layout()
out_file = Path('../results/figures/fig_era5_systemic_risk_baseline.png')
out_file.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(out_file, dpi=300, bbox_inches='tight')
plt.show()

print(f"\n✓ Figure saved to: {out_file}")


## 3. Policy Scenario Comparison

Compare systemic risk probabilities under different policy interventions:
- **Baseline**: No intervention
- **Market Exit (Moderate)**: 10% of high-risk properties exit private market
- **Penetration (Major)**: 40% increase in flood insurance take-up
- **Building Codes (Major)**: 20% damage reduction from stricter codes

### 3.1 Load Policy Scenario Data

Load iterations.csv for each policy scenario run.

### Policy Scenario Parametrization

The three policy scenarios analyzed represent different adaptation pathways based on actual code implementation from December 2025 runs:

**1. Market Exit (Moderate)** - Private insurer market withdrawal
- **Citizens market share**: 15% → 25% (+10 percentage points)
- **Exit mechanism**: Uniform reduction across all private insurers
- **Citizens absorption rate**: 85% of exited exposure → Citizens, 15% → uninsured
- **What it modifies**: 
  - ✅ Reallocates TIV from private insurers to Citizens (pre-event)
  - ✅ Adjusts group capital (surplus) for companies that lose exposure
  - ✅ Scales Citizens capital needs based on increased exposure share
  - Does NOT modify wind/flood losses directly

**2. Penetration (Major)** - Insurance take-up increase
- **Wind penetration**: 40% → 60% (+20 percentage points = 50% relative increase)
- **Flood penetration**: 11% → 30% (+19 percentage points = 173% relative increase)
- **Citizens share**: 15% → 8% (-7 percentage points, depopulation to private market)
- **Geographic targeting**: 
  - Coastal counties: 1.5× higher increase vs. inland
  - SFHA-aware flood scaling: 3× growth in non-SFHA zones, 1.2× in SFHA zones
- **What it modifies**: 
  - ✅ Increases TIV for NFIP (flood coverage, pre-event)
  - ✅ Increases TIV for private insurers (wind coverage, pre-event)
  - ✅ Transfers exposure from Citizens to private market
  - ✅ Scales surplus (capital) proportionally with exposure growth
  - Does NOT modify wind/flood losses directly

**3. Building Codes (Major)** - Loss reduction from improved construction
- **Wind loss reduction**: 30% (applied to all wind losses)
- **Flood loss reduction**: 25% (applied to all flood losses)
- **What it modifies**: 
  - ✅ Reduces **wind losses** by 30% (applied AFTER impact calculation, BEFORE carveout)
  - ✅ Reduces **flood losses** by 25% (applied AFTER impact calculation, BEFORE carveout)
  - Applied to gross damages before insured/underinsured/uninsured allocation
  - Does NOT modify exposure (TIV) or capital
- **Evidence base**: IBHS FORTIFIED (midpoint), FLASH study modern codes

**Key Implementation Notes:**
- Market Exit and Penetration modify **exposure allocation** (pre-event)
- Building Codes modifies **loss magnitude** (post-impact, pre-carveout)
- All scenarios use deterministic parameters (no stochastic variation within scenario)
- Capital adjustments use "proportional" scaling (stress ratios maintained)
- Actual December 2025 runs: `emanuel_era5_market_exit_moderate_20251207_074006`, `emanuel_era5_penetration_major_20251207_074022`, `emanuel_era5_building_codes_major_20251207_074022`

In [ ]:
# Load policy scenario iterations
# NOTE: Update directory timestamps after Sherlock runs complete
# Use glob to find the most recent runs automatically

import glob

def find_latest_run(pattern):
    """Find the most recent MC run matching pattern."""
    matches = glob.glob(f'../results/mc_runs/{pattern}')
    if not matches:
        return None
    return max(matches, key=lambda p: Path(p).stat().st_mtime)

# Try to load policy scenarios
policy_scenarios = {}

# Baseline
# Replace find_latest_run() with the explicit path from section 2
#baseline_dir = '../results/mc_runs/emanuel_era5_baseline_20251207_072016'
baseline_dir = '../results/mc_runs/emanuel_era5_baseline_20260210_122648'
if baseline_dir:
    policy_scenarios['Baseline'] = pd.read_csv(f'{baseline_dir}/iterations.csv')
    print(f"✓ Loaded Baseline: {baseline_dir}")

# Market Exit
market_exit_dir = find_latest_run('emanuel_era5_market_exit_moderate_*')
if market_exit_dir:
    policy_scenarios['Market Exit'] = pd.read_csv(f'{market_exit_dir}/iterations.csv')
    print(f"✓ Loaded Market Exit: {market_exit_dir}")

# Penetration (FIXED: was overwriting with market_exit_dir)
penetration_dir = find_latest_run('emanuel_era5_penetration_major_*')
#penetration_dir = find_latest_run('emanuel_era5_market_exit_moderate_*')
if penetration_dir:
    policy_scenarios['Penetration'] = pd.read_csv(f'{penetration_dir}/iterations.csv')
    print(f"✓ Loaded Penetration: {penetration_dir}")

# Building Codes
# Building Codes
#building_codes_dir = find_latest_run('era5_building_codes_major_rerun_20260205_150855')
building_codes_dir = find_latest_run('emanuel_era5_building_codes_major_20260210_122648')
#building_codes_dir = find_latest_run('era5_building_codes_major_*')
if building_codes_dir:
    policy_scenarios['Building Codes'] = pd.read_csv(f'{building_codes_dir}/iterations.csv')
    print(f"✓ Loaded Building Codes: {building_codes_dir}")

print(f"\n{len(policy_scenarios)} policy scenarios loaded")
if len(policy_scenarios) == 0:
    print("⚠️  No policy scenario data found. Run policy jobs on Sherlock first.")

### 3.2 Calculate Policy Scenario Metrics

Compute annual exceedance probabilities for each policy scenario.

In [ ]:
# Only proceed if we have policy data
if len(policy_scenarios) > 0:
    # Calculate metrics for each policy scenario directly from 10,000 MC years
    policy_metrics = {}
    policy_metrics_p10 = {}
    policy_metrics_p90 = {}

    for scenario_name, df_scenario in policy_scenarios.items():
        # Filter out errors
        df_valid_policy = df_scenario[df_scenario['scenario'] != 'error'].copy()
        
        metrics = {}
        metrics_p10 = {}
        metrics_p90 = {}
        
        # Private market metrics
        defaults_10 = (df_valid_policy['defaults_post'] > 10)
        unc = compute_uncertainty_bounds(defaults_10)
        metrics['Defaults > 10'] = unc['mean']
        metrics_p10['Defaults > 10'] = unc['p10']
        metrics_p90['Defaults > 10'] = unc['p90']
        
        deficit_1b = (df_valid_policy['largest_entity_deficit_usd'] > 1e9)
        unc = compute_uncertainty_bounds(deficit_1b)
        metrics['Single Deficit > $1B'] = unc['mean']
        metrics_p10['Single Deficit > $1B'] = unc['p10']
        metrics_p90['Single Deficit > $1B'] = unc['p90']
        
        # Institutional stress metrics
        fhcf_cap_exceeded = (df_valid_policy['fhcf_recovery_private_usd'] + df_valid_policy['fhcf_recovery_citizens_usd']) / FHCF_CAP > 1.0
        unc = compute_uncertainty_bounds(fhcf_cap_exceeded)
        metrics['FHCF > 100% Cap'] = unc['mean']
        metrics_p10['FHCF > 100% Cap'] = unc['p10']
        metrics_p90['FHCF > 100% Cap'] = unc['p90']
        
        figa_capacity_exceeded = df_valid_policy['figa_residual_deficit_usd'] > 0
        unc = compute_uncertainty_bounds(figa_capacity_exceeded)
        metrics['FIGA > 100% Capacity'] = unc['mean']
        metrics_p10['FIGA > 100% Capacity'] = unc['p10']
        metrics_p90['FIGA > 100% Capacity'] = unc['p90']
        
        citizens_capacity_exceeded = df_valid_policy['citizens_residual_deficit_usd'] / (df_valid_policy['citizens_tier1_capacity_usd'] + df_valid_policy['citizens_tier2_capacity_usd']) > 1.0
        unc = compute_uncertainty_bounds(citizens_capacity_exceeded)
        metrics['Citizens > 100% Capacity'] = unc['mean']
        metrics_p10['Citizens > 100% Capacity'] = unc['p10']
        metrics_p90['Citizens > 100% Capacity'] = unc['p90']
        
        nfip_stressed = df_valid_policy['nfip_claims_paid_usd'] / df_valid_policy['nfip_fl_premium_base_usd'] > 2.0
        unc = compute_uncertainty_bounds(nfip_stressed)
        metrics['NFIP > 200% Annual Premium'] = unc['mean']
        metrics_p10['NFIP > 200% Annual Premium'] = unc['p10']
        metrics_p90['NFIP > 200% Annual Premium'] = unc['p90']
        
        # Aggregate fiscal impact metrics (relative to Florida GDP)
        total_public_burden_policy = (
            df_valid_policy['fhcf_shortfall_usd'] + 
            df_valid_policy['figa_residual_deficit_usd'] + 
            df_valid_policy['citizens_residual_deficit_usd'] + 
            df_valid_policy['nfip_borrowed_usd']
        )
        
        FL_GDP = 1.7e12  # $1.7 trillion (2024-25)
        
        pub_burden_1pct = (total_public_burden_policy > 0.01 * FL_GDP)
        unc = compute_uncertainty_bounds(pub_burden_1pct)
        metrics['Public Burden > 1% FL GDP'] = unc['mean']
        metrics_p10['Public Burden > 1% FL GDP'] = unc['p10']
        metrics_p90['Public Burden > 1% FL GDP'] = unc['p90']
        
        pub_burden_10pct = (total_public_burden_policy > 0.10 * FL_GDP)
        unc = compute_uncertainty_bounds(pub_burden_10pct)
        metrics['Public Burden > 10% FL GDP'] = unc['mean']
        metrics_p10['Public Burden > 10% FL GDP'] = unc['p10']
        metrics_p90['Public Burden > 10% FL GDP'] = unc['p90']
        
        policy_metrics[scenario_name] = metrics
        policy_metrics_p10[scenario_name] = metrics_p10
        policy_metrics_p90[scenario_name] = metrics_p90
    
    # Create comparison DataFrame
    # Order: Baseline, Market Exit, Penetration, Building Codes
    scenario_order = ['Baseline', 'Market Exit', 'Penetration', 'Building Codes']
    available_scenarios = [s for s in scenario_order if s in policy_metrics]
    
    # Build dataframe with mean and uncertainty bounds
    df_policy_comparison = pd.DataFrame({
        'Metric': metric_names
    })
    
    for scenario in available_scenarios:
        df_policy_comparison[scenario] = [policy_metrics[scenario][m] for m in metric_names]
        df_policy_comparison[f'{scenario}_P10'] = [policy_metrics_p10[scenario][m] for m in metric_names]
        df_policy_comparison[f'{scenario}_P90'] = [policy_metrics_p90[scenario][m] for m in metric_names]
    
    # Reverse for display
    df_policy_comparison_display = df_policy_comparison[::-1].reset_index(drop=True)
    
    print("\n" + "="*90)
    print("SYSTEMIC RISK PROBABILITIES: POLICY SCENARIO COMPARISON")
    print("="*90)
    
    # Display only mean values for table
    display_cols = ['Metric'] + available_scenarios
    print(df_policy_comparison_display[display_cols].to_string(index=False))
    print("="*90)
else:
    print("\n⚠️  Skipping policy comparison - no policy data available yet")


### 3.3 Visualization: Policy Comparison

Grouped bar chart comparing annual probabilities across policy scenarios.

In [ ]:
# Alternative visualization: Divergence from baseline with diverging bar plot
if len(policy_scenarios) >= 2 and 'Baseline' in available_scenarios:
    # Calculate divergence from baseline for each scenario
    baseline_vals = df_policy_comparison_display['Baseline'].values
    baseline_p10 = df_policy_comparison_display['Baseline_P10'].values
    baseline_p90 = df_policy_comparison_display['Baseline_P90'].values
    
    # Get non-baseline scenarios (reversed so Market Exit is at top)
    policy_scenarios_only = [s for s in available_scenarios if s != 'Baseline']
    policy_scenarios_only = policy_scenarios_only[::-1]  # Reverse order
    
    if len(policy_scenarios_only) > 0:
        fig, ax = plt.subplots(figsize=(5.5,4))
        
        # Prepare data
        y_pos = np.arange(len(df_policy_comparison_display))
        n_scenarios = len(policy_scenarios_only)
        bar_height = 0.7 / n_scenarios  # Height of each bar
        
        # Define colors for each scenario (excluding baseline)
        colors_map = {
            'Market Exit': '#10B981',    # Green
            'Penetration': '#F59E0B',    # Amber
            'Building Codes': '#3B82F6'  # Blue
        }
        
        # Plot each policy scenario's divergence as horizontal bars
        for i, scenario in enumerate(policy_scenarios_only):
            # Get scenario values and uncertainty
            scenario_vals = df_policy_comparison_display[scenario].values
            scenario_p10 = df_policy_comparison_display[f'{scenario}_P10'].values
            scenario_p90 = df_policy_comparison_display[f'{scenario}_P90'].values
            
            # Calculate divergence (percentage points)
            divergence = scenario_vals - baseline_vals
            
            # Propagate uncertainty: for differences, uncertainties combine
            # Lower bound of difference: scenario_p10 - baseline_p90
            # Upper bound of difference: scenario_p90 - baseline_p10
            divergence_lower = scenario_p10 - baseline_p90
            divergence_upper = scenario_p90 - baseline_p10
            
            # Calculate error bar values
            lower_errors = divergence - divergence_lower
            upper_errors = divergence_upper - divergence
            
            # Position offset for each scenario
            y_offset = (i - (n_scenarios - 1) / 2) * bar_height
            
            # Plot diverging bars with error bars
            ax.barh(y_pos + y_offset, divergence, bar_height,
                   xerr=[lower_errors, upper_errors],
                   color=colors_map[scenario], edgecolor='black', linewidth=0.8,
                   label=scenario,
                   error_kw={'linewidth': 1.2, 'ecolor': 'black', 'capsize': 3, 'capthick': 1.2})
        
        # Add vertical line at zero (baseline reference)
        ax.axvline(x=0, color='k', linewidth=1.0, linestyle='-', zorder=1)
        
        # Add category separators (extended beyond plot area)
        ax.axhline(y=1.5, color='black', linewidth=1.5, linestyle='-', 
                   xmin=-1.5, xmax=1.0, clip_on=False)
        ax.axhline(y=5.5, color='black', linewidth=1.5, linestyle='-',
                   xmin=-1.5, xmax=1.0, clip_on=False)
        
        ax.set_xlabel('Change in annual probability relative to ERA5 (percentage points)', fontsize=11, fontweight='normal')
        ax.set_ylabel('')
        
        # Set y-axis
        ax.set_yticks(y_pos)
        ax.set_yticklabels(df_policy_comparison_display['Metric'], fontsize=9)
        
        # Set x-axis limits: cap left side at -3, calculate right side dynamically
        all_divergences = []
        for scenario in policy_scenarios_only:
            scenario_vals = df_policy_comparison_display[scenario].values
            scenario_p10 = df_policy_comparison_display[f'{scenario}_P10'].values
            scenario_p90 = df_policy_comparison_display[f'{scenario}_P90'].values
            all_divergences.extend((scenario_vals - baseline_vals).tolist())
            all_divergences.extend((scenario_p10 - baseline_p90).tolist())
            all_divergences.extend((scenario_p90 - baseline_p10).tolist())
        
        if all_divergences:
            max_positive = max(all_divergences)
            ax.set_xlim(-3, max_positive * 1.15)
        else:
            ax.set_xlim(-3, 3)
            #ax.set_xlim(-3, 16)
        
        # Remove all grids
        ax.grid(False)
        
        # Style axes
        ax.spines['left'].set_color('black')
        ax.spines['bottom'].set_color('black')
        ax.spines['left'].set_linewidth(1)
        ax.spines['bottom'].set_linewidth(1)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.tick_params(axis='both', colors='black')
        ax.tick_params(axis='y', labelsize=9)
        
        # Add legend on the right side
        ax.legend(loc='center left', frameon=True, fancybox=False, 
                  edgecolor='black', framealpha=0.9, fontsize=9, 
                  ncol=1, bbox_to_anchor=(1.02, 0.5))
        
        # Add category labels
        fig.text(0.01, 0.85, 'PRIVATE\nMARKET', fontsize=8, fontweight='normal', 
                 va='center', ha='center', color='k', rotation=90)
        fig.text(0.01, 0.57, 'INSTITUTIONAL\nSTRESS', fontsize=8, fontweight='normal', 
                 va='center', ha='center', color='k', rotation=90)
        fig.text(0.01, 0.25, 'AGG. FISCAL\nIMPACT', fontsize=8, fontweight='normal', 
                 va='center', ha='center', color='k', rotation=90)
        
        plt.tight_layout()
        out_file = Path('../results/figures/fig_era5_policy_divergence.png')
        plt.savefig(out_file, dpi=300, bbox_inches='tight')
        plt.show()
        
        print(f"\n✓ Policy divergence figure saved to: {out_file}")
        print(f"   Showing divergence from baseline for {len(policy_scenarios_only)} policy scenarios")
        print(f"   Positive values = increased risk; Negative values = reduced risk")
    else:
        print("\n⚠️  No policy scenarios available besides baseline")
else:
    print("\n⚠️  Need baseline + policy scenarios to create divergence plot")

In [ ]:
# Create results/tables directory
tables_dir = Path('../results/tables')
tables_dir.mkdir(parents=True, exist_ok=True)

# Export ERA5 baseline metrics
df_era5_metrics.to_csv(tables_dir / 'era5_baseline_probabilities.csv', index=False)
print(f"✓ Saved: {tables_dir / 'era5_baseline_probabilities.csv'}")

# Export policy comparison if available
if len(policy_scenarios) >= 2:
    df_policy_comparison.to_csv(tables_dir / 'era5_policy_comparison.csv', index=False)
    print(f"✓ Saved: {tables_dir / 'era5_policy_comparison.csv'}")
    
print("\n✓ All tables exported")

## 5. Climate Change Projections

Analyze how climate change affects systemic risk using GCM ensemble projections.

**Approach:**
1. Load climate-scaled ERA5 results (from `compute_climate_deltas.py`)
2. Compare baseline vs near-term (2041-2060) vs mid-term (2081-2100)
3. Compare SSP2-4.5 vs SSP5-8.5 pathways
4. Show GCM ensemble uncertainty (p10-p90 range)

### 5.1 Load Climate Delta Results

First, run the climate delta analysis script (after all GCM runs complete):

```bash
python fl_risk_model/analysis/compute_climate_deltas.py \
    --mc_root results/mc_runs \
    --out results/climate_deltas
```

Then load the scaled ERA5 results.

In [ ]:
# Check if climate delta files exist
climate_dir = Path('../results/climate_deltas')

if not climate_dir.exists():
    print("⚠️  Climate delta directory not found")
    print("    Run: python fl_risk_model/analysis/compute_climate_deltas.py \\")
    print("              --mc_root results/mc_runs --out results/climate_deltas")
    climate_data_available = False
else:
    # Check for required files
    required_files = [
        'era5_climate_scaled_absolute.csv',
        'era5_climate_scaled_relative.csv',
        'gcm_era5_alignment.csv',
        'climate_deltas_ensemble_absolute.csv'
    ]
    
    missing_files = [f for f in required_files if not (climate_dir / f).exists()]
    
    if missing_files:
        print(f"⚠️  Missing files: {missing_files}")
        print("    Run climate delta analysis script first")
        climate_data_available = False
    else:
        print("✓ Climate delta files found")
        climate_data_available = True
        
        # Load the data
        df_era5_scaled_abs = pd.read_csv(climate_dir / 'era5_climate_scaled_absolute.csv')
        df_era5_scaled_rel = pd.read_csv(climate_dir / 'era5_climate_scaled_relative.csv')
        df_gcm_alignment = pd.read_csv(climate_dir / 'gcm_era5_alignment.csv')
        df_deltas_ensemble = pd.read_csv(climate_dir / 'climate_deltas_ensemble_absolute.csv')
        
        print(f"\n✓ Loaded climate projections:")
        print(f"  - ERA5 scaled (absolute): {len(df_era5_scaled_abs)} rows")
        print(f"  - ERA5 scaled (relative): {len(df_era5_scaled_rel)} rows")
        print(f"  - GCM alignment: {len(df_gcm_alignment)} rows")
        print(f"  - Ensemble deltas: {len(df_deltas_ensemble)} rows")
        
        print(f"\n  Time categories: {df_era5_scaled_abs['time_category'].unique()}")

### 5.3 Climate Change Impact on Systemic Risk

Calculate systemic risk probabilities under future climate scenarios.

For this analysis, we'll work with return period summary data rather than individual iterations,
as the climate delta analysis scales return period metrics.

In [ ]:
if climate_data_available:
    # Use the ABSOLUTE delta approach (recommended to avoid GCM bias amplification)
    df_climate = df_era5_scaled_abs.copy()
    
    print("\n" + "="*80)
    print("CLIMATE CHANGE IMPACT SUMMARY")
    print("="*80)
    print("Using ABSOLUTE deltas (additive method) to avoid GCM baseline bias amplification")
    print("="*80)
    
    # Define key systemic risk metrics (matching Section 2)
    key_metrics = [
        'total_damage_usd',
        'defaults_post', 
        'largest_entity_deficit_usd',
        'fhcf_recovery_private_usd',
        'fhcf_recovery_citizens_usd',
        'figa_residual_deficit_usd',
        'citizens_residual_deficit_usd',
        'nfip_borrowed_usd',
        'fhcf_shortfall_usd'
    ]
    
    # Check which metrics are available
    available_metrics = [m for m in key_metrics if m in df_climate['metric'].values]
    
    print(f"\nFound {len(available_metrics)} key systemic risk metrics")
    
    if len(available_metrics) > 0:
        # Data is already at annual average level (no return periods)
        # Extract values for each time period
        
        results = []
        
        for time_cat in ['near', 'mid']:
            for metric in available_metrics:
                metric_data = df_climate[
                    (df_climate['metric'] == metric) &
                    (df_climate['time_category'] == time_cat)
                ]
                
                if len(metric_data) == 0:
                    continue
                
                row = metric_data.iloc[0]
                baseline = row['era5_baseline']
                scaled = row['scaled_median']
                scaled_p10 = row['scaled_p10']
                scaled_p90 = row['scaled_p90']
                
                pct_change = ((scaled - baseline) / baseline * 100) if baseline > 0 else 0
                
                results.append({
                    'time_category': time_cat,
                    'metric': metric,
                    'baseline': baseline,
                    'scaled_median': scaled,
                    'scaled_p10': scaled_p10,
                    'scaled_p90': scaled_p90,
                    'pct_change': pct_change
                })
        
        df_climate_summary = pd.DataFrame(results)
        
        print(f"\nAnnual Average Analysis:")
        print(f"Data for {len(df_climate_summary)} metric-period combinations")
        
        # Display summary by time period
        print("\n" + "-"*80)
        print("Key Metrics Change:")
        print("-"*80)
        
        display_metrics = ['fhcf_shortfall_usd', 'figa_residual_deficit_usd', 'citizens_residual_deficit_usd', 'total_damage_usd']
        
        for time_cat in ['near', 'mid']:
            time_label = '2041-2060' if time_cat == 'near' else '2081-2100'
            print(f"\n{time_label}:")
            
            for metric in display_metrics:
                data = df_climate_summary[
                    (df_climate_summary['metric'] == metric) &
                    (df_climate_summary['time_category'] == time_cat)
                ]
                if not data.empty:
                    row = data.iloc[0]
                    metric_name = metric.replace('_usd', '').replace('_', ' ')
                    print(f"  {metric_name:30s}: ${row['baseline']/1e9:5.2f}B → ${row['scaled_median']/1e9:5.2f}B ({row['pct_change']:+5.1f}%)")
        
        print("-"*80)
    else:
        print("\n⚠️  No matching metrics found in climate delta output")
        print(f"    Available metrics: {df_climate['metric'].unique()[:10]}")
else:
    print("⚠️  Skipping climate impact analysis - data not available")

### 5.5 Multi-Scenario Climate Comparison

Horizontal bar plot comparing baseline with 4 future scenarios:
- **2050 SSP2-4.5** (near-term, moderate emissions)
- **2050 SSP5-8.5** (near-term, high emissions)  
- **2100 SSP2-4.5** (mid-century, moderate emissions)
- **2100 SSP5-8.5** (mid-century, high emissions)

Error bars show GCM ensemble uncertainty (p10-p90 range).

In [ ]:
from matplotlib.patches import Patch

# ---- Desired top-to-bottom order ----
desired_order = [
    # PRIVATE MARKET
    'Defaults > 10',
    'Single Deficit > $1B',

    # INSTITUTIONAL CAPACITY
    'FHCF > 100% Cap',
    'FIGA > 100% Capacity',
    'Citizens > 100% Capacity',
    'NFIP > 200% Annual Premium',

    # AGGREGATE FISCAL IMPACT
    'Public Burden > 1% FL GDP',
    'Public Burden > 10% FL GDP'
]

df_climate_comparison = (
    df_climate_comparison
    .set_index('Metric')
    .reindex(desired_order)
    .reset_index()
)

# Create visualization with uncertainty bars
fig, ax = plt.subplots(figsize=(7, 4.5))

# Metric order now explicitly controlled above
y_pos = np.arange(len(df_climate_comparison))
n_scenarios = 5
width = 0.75 / n_scenarios

# Extract median values
baseline_vals    = df_climate_comparison['Baseline'].values
near_ssp245_vals = df_climate_comparison['2050 SSP2-4.5'].values
near_ssp585_vals = df_climate_comparison['2050 SSP5-8.5'].values
mid_ssp245_vals  = df_climate_comparison['2100 SSP2-4.5'].values
mid_ssp585_vals  = df_climate_comparison['2100 SSP5-8.5'].values

# Calculate error bars (asymmetric)
def get_error_bars(df, col_prefix):
    """Get asymmetric error bars (lower, upper)."""
    medians = df[col_prefix].values
    p10s = df.get(f'{col_prefix}_p10', medians)
    p90s = df.get(f'{col_prefix}_p90', medians)

    lower = np.maximum(medians - p10s, 0)  # Don't go below 0
    upper = np.maximum(p90s - medians, 0)
    return [lower, upper]

near_245_err = get_error_bars(df_climate_comparison, '2050 SSP2-4.5')
near_585_err = get_error_bars(df_climate_comparison, '2050 SSP5-8.5')
mid_245_err  = get_error_bars(df_climate_comparison, '2100 SSP2-4.5')
mid_585_err  = get_error_bars(df_climate_comparison, '2100 SSP5-8.5')
baseline_err = get_error_bars(df_climate_comparison, 'Baseline')

# ---- Colors: 3 categories with variants ----
color_baseline  = "#6B7280"  # neutral gray

# Mid-century (2050): same family, two shades
color_2050_245  = '#FBBF24'  # lighter amber
color_2050_585  = '#D97706'  # darker amber

# End-century (2100): same family, two shades
color_2100_245  = '#FCA5A5'  # lighter red
color_2100_585  = '#B91C1C'  # darker red

# ---- Bar positions: baseline at top, then 2050, then 2100 ----
# Top -> bottom within each metric row:
# Baseline (highest), 2050 SSP2-4.5, 2050 SSP5-8.5, 2100 SSP2-4.5, 2100 SSP5-8.5
#offsets = np.array([2*width, 1*width, 0, -1*width, -2*width])
offsets = np.array([-2*width, -1*width, 0, 1*width, 2*width])

# Create bars with error bars
ax.barh(
    y_pos + offsets[0], baseline_vals, width,
    xerr=baseline_err, color=color_baseline, edgecolor='black', linewidth=0.8,
    alpha=0.8,
    error_kw={'linewidth': 1.2, 'ecolor': 'black', 'capsize': 2, 'capthick': 1.2}
)

ax.barh(
    y_pos + offsets[1], near_ssp245_vals, width,
    xerr=near_245_err, color=color_2050_245, edgecolor='black', linewidth=0.8,
    alpha=0.9,
    error_kw={'linewidth': 1.2, 'ecolor': 'black', 'capsize': 2, 'capthick': 1.2}
)

ax.barh(
    y_pos + offsets[2], near_ssp585_vals, width,
    xerr=near_585_err, color=color_2050_585, edgecolor='black', linewidth=0.8,
    alpha=0.9,
    error_kw={'linewidth': 1.2, 'ecolor': 'black', 'capsize': 2, 'capthick': 1.2}
)

ax.barh(
    y_pos + offsets[3], mid_ssp245_vals, width,
    xerr=mid_245_err, color=color_2100_245, edgecolor='black', linewidth=0.8,
    alpha=0.95,
    error_kw={'linewidth': 1.2, 'ecolor': 'black', 'capsize': 2, 'capthick': 1.2}
)

ax.barh(
    y_pos + offsets[4], mid_ssp585_vals, width,
    xerr=mid_585_err, color=color_2100_585, edgecolor='black', linewidth=0.8,
    alpha=0.95,
    error_kw={'linewidth': 1.2, 'ecolor': 'black', 'capsize': 2, 'capthick': 1.2}
)

# ---- Axis limits (robust to NaN/Inf) ----
all_vals = np.concatenate([
    baseline_vals, near_ssp245_vals, near_ssp585_vals,
    mid_ssp245_vals, mid_ssp585_vals
])

finite_all = all_vals[np.isfinite(all_vals)]
if finite_all.size == 0:
    max_val = 1.0  # fallback if everything is NaN
else:
    max_val = finite_all.max()

if '2100 SSP5-8.5_p90' in df_climate_comparison.columns:
    p90_vals = df_climate_comparison['2100 SSP5-8.5_p90'].values
    finite_p90 = p90_vals[np.isfinite(p90_vals)]
    if finite_p90.size > 0:
        max_val = max(max_val, finite_p90.max())

# Final sanity fallback (in case max_val is still non-finite)
if not np.isfinite(max_val) or max_val <= 0:
    max_val = 1.0

ax.set_xlim(0, max_val * 1.25)

# ---- Category separators (for 2 / 4 / 2 rows) ----
# After index 1 (2 private metrics) and after index 5 (4 institutional metrics)
if len(df_climate_comparison) >= 8:
    ax.axhline(y=1.5, color='black', linewidth=1.5, xmin=-1.5, xmax=1.0, clip_on=False)
    ax.axhline(y=5.5, color='black', linewidth=1.5, xmin=-1.5, xmax=1.0, clip_on=False)

# Formatting
ax.set_xlabel('Annual Probability (%)', fontsize=11, fontweight='normal')
ax.set_yticks(y_pos)
ax.set_yticklabels(df_climate_comparison['Metric'], fontsize=9)
ax.invert_yaxis()  # <- this makes the desired_order appear top-to-bottom

ax.grid(False)

ax.spines['left'].set_color('black')
ax.spines['bottom'].set_color('black')
ax.spines['left'].set_linewidth(1)
ax.spines['bottom'].set_linewidth(1)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.tick_params(axis='both', colors='black')
ax.tick_params(axis='y', labelsize=9)

# ---- Legend as bbox center-right ----
legend_handles = [
    Patch(facecolor=color_baseline,  edgecolor='black', label='ERA5'),
    Patch(facecolor=color_2050_245, edgecolor='black', label='2050 SSP2-4.5'),
    Patch(facecolor=color_2050_585, edgecolor='black', label='2050 SSP5-8.5'),
    Patch(facecolor=color_2100_245, edgecolor='black', label='2100 SSP2-4.5'),
    Patch(facecolor=color_2100_585, edgecolor='black', label='2100 SSP5-8.5'),
]

ax.legend(
    handles=legend_handles,
    loc='center left',
    bbox_to_anchor=(1.02, 0.5),
    frameon=True,
    fancybox=False,
    edgecolor='black',
    framealpha=0.95,
    fontsize=8,
)

# Category labels (positions tuned for 8 rows)
if len(df_climate_comparison) >= 8:
    fig.text(0.00, 0.85, 'PRIVATE\nMARKET', fontsize=8, fontweight='normal',
             va='center', ha='center', color='k', rotation=90)
    fig.text(0.00, 0.57, 'INSTITUTIONAL\nSTRESS', fontsize=8, fontweight='normal',
             va='center', ha='center', color='k', rotation=90)
    fig.text(0.00, 0.25, 'AGG. FISCAL\nIMPACT', fontsize=8, fontweight='normal',
             va='center', ha='center', color='k', rotation=90)

plt.tight_layout()

out_file = Path('../results/figures/fig_climate_systemic_risk_scenarios.png')
plt.savefig(out_file, dpi=300, bbox_inches='tight')
plt.show()

print(f"\n✓ Multi-scenario systemic risk figure saved to: {out_file}")


In [ ]:
# Export climate comparison if available
if 'df_climate_comparison' in dir() and df_climate_comparison is not None and not df_climate_comparison.empty:
    df_climate_comparison.to_csv(tables_dir / 'era5_climate_comparison.csv', index=False)
    print(f"✓ Saved: {tables_dir / 'era5_climate_comparison.csv'}")

## 4. Summary Tables

Export summary statistics to CSV files for reporting.

In [ ]:
# Prepare loss decomposition data for different scenarios
# We'll show: Baseline, 4 climate scenarios, and available policy scenarios

loss_data = {}
institutional_data = {}

# ===== BASELINE (ERA5) =====
# Calculate mean and percentiles for baseline
baseline_losses = {
    'insured_wind': df_valid['wind_insured_private_usd'] / 1e9,
    'citizens': df_valid['wind_insured_citizens_usd'] / 1e9,
    'insured_flood': df_valid['flood_insured_capped_usd'] / 1e9,
    'wind_under': (df_valid['wind_total_usd'] - 
                   df_valid['wind_insured_private_usd'] - 
                   df_valid['wind_insured_citizens_usd']) / 1e9,
    'flood_under': (df_valid['water_total_usd'] - 
                    df_valid['flood_insured_capped_usd']) / 1e9
}

loss_data['Baseline\n(ERA5)'] = {
    'insured_wind': baseline_losses['insured_wind'].mean(),
    'insured_wind_p10': baseline_losses['insured_wind'].quantile(0.10),
    'insured_wind_p90': baseline_losses['insured_wind'].quantile(0.90),
    'citizens': baseline_losses['citizens'].mean(),
    'citizens_p10': baseline_losses['citizens'].quantile(0.10),
    'citizens_p90': baseline_losses['citizens'].quantile(0.90),
    'insured_flood': baseline_losses['insured_flood'].mean(),
    'insured_flood_p10': baseline_losses['insured_flood'].quantile(0.10),
    'insured_flood_p90': baseline_losses['insured_flood'].quantile(0.90),
    'wind_under': baseline_losses['wind_under'].mean(),
    'wind_under_p10': baseline_losses['wind_under'].quantile(0.10),
    'wind_under_p90': baseline_losses['wind_under'].quantile(0.90),
    'flood_under': baseline_losses['flood_under'].mean(),
    'flood_under_p10': baseline_losses['flood_under'].quantile(0.10),
    'flood_under_p90': baseline_losses['flood_under'].quantile(0.90),
}

# Add total losses
total_loss = sum([baseline_losses[k].mean() for k in ['insured_wind', 'citizens', 'insured_flood', 'wind_under', 'flood_under']])
total_loss_p10 = sum([baseline_losses[k].quantile(0.10) for k in ['insured_wind', 'citizens', 'insured_flood', 'wind_under', 'flood_under']])
total_loss_p90 = sum([baseline_losses[k].quantile(0.90) for k in ['insured_wind', 'citizens', 'insured_flood', 'wind_under', 'flood_under']])

loss_data['Baseline\n(ERA5)']['total_loss'] = total_loss
loss_data['Baseline\n(ERA5)']['total_loss_p10'] = total_loss_p10
loss_data['Baseline\n(ERA5)']['total_loss_p90'] = total_loss_p90

# Calculate institutional data with uncertainties
baseline_inst = {
    'fhcf_shortfall': df_valid['fhcf_shortfall_usd'] / 1e9,
    'figa_residual': df_valid['figa_residual_deficit_usd'] / 1e9,
    'citizens_deficit': df_valid['citizens_residual_deficit_usd'] / 1e9,
    'nfip_borrowed': df_valid['nfip_borrowed_usd'] / 1e9
}

institutional_data['Baseline\n(ERA5)'] = {
    'fhcf_shortfall': baseline_inst['fhcf_shortfall'].mean(),
    'fhcf_shortfall_p10': baseline_inst['fhcf_shortfall'].quantile(0.10),
    'fhcf_shortfall_p90': baseline_inst['fhcf_shortfall'].quantile(0.90),
    'figa_residual': baseline_inst['figa_residual'].mean(),
    'figa_residual_p10': baseline_inst['figa_residual'].quantile(0.10),
    'figa_residual_p90': baseline_inst['figa_residual'].quantile(0.90),
    'citizens_deficit': baseline_inst['citizens_deficit'].mean(),
    'citizens_deficit_p10': baseline_inst['citizens_deficit'].quantile(0.10),
    'citizens_deficit_p90': baseline_inst['citizens_deficit'].quantile(0.90),
    'nfip_borrowed': baseline_inst['nfip_borrowed'].mean(),
    'nfip_borrowed_p10': baseline_inst['nfip_borrowed'].quantile(0.10),
    'nfip_borrowed_p90': baseline_inst['nfip_borrowed'].quantile(0.90),
}

# ===== CLIMATE SCENARIOS (if available) =====
# Check if climate data exists
if 'df_climate' in dir():
    climate_scenarios_list = [
        ('2050\nSSP2-4.5', 'ssp245', 'near'),
        ('2050\nSSP5-8.5', 'ssp585', 'near'),
        ('2100\nSSP2-4.5', 'ssp245', 'mid'),
        ('2100\nSSP5-8.5', 'ssp585', 'mid')
    ]
    
    for label, pathway, time_cat in climate_scenarios_list:
        # Get scaled values from df_climate for all relevant metrics
        climate_subset = df_climate[
            (df_climate['pathway'] == pathway) &
            (df_climate['time_category'] == time_cat)
        ]
        
        if not climate_subset.empty:
            # Helper function to get scaled values with uncertainties
            def get_scaled_values(metric_name):
                metric_row = climate_subset[climate_subset['metric'] == metric_name]
                if not metric_row.empty:
                    row = metric_row.iloc[0]
                    return {
                        'mean': row['scaled_median'] / 1e9,
                        'p10': row['scaled_p10'] / 1e9 if 'scaled_p10' in row else row['scaled_median'] / 1e9,
                        'p90': row['scaled_p90'] / 1e9 if 'scaled_p90' in row else row['scaled_median'] / 1e9,
                    }
                return {'mean': 0.0, 'p10': 0.0, 'p90': 0.0}
            
            # Extract loss components
            insured_wind = get_scaled_values('wind_insured_private_usd')
            citizens = get_scaled_values('wind_insured_citizens_usd')
            insured_flood = get_scaled_values('flood_insured_capped_usd')
            wind_under = get_scaled_values('wind_underinsured_usd')
            flood_under = get_scaled_values('flood_un_derinsured_usd')
            
            loss_data[label] = {
                'insured_wind': insured_wind['mean'],
                'insured_wind_p10': insured_wind['p10'],
                'insured_wind_p90': insured_wind['p90'],
                'citizens': citizens['mean'],
                'citizens_p10': citizens['p10'],
                'citizens_p90': citizens['p90'],
                'insured_flood': insured_flood['mean'],
                'insured_flood_p10': insured_flood['p10'],
                'insured_flood_p90': insured_flood['p90'],
                'wind_under': wind_under['mean'],
                'wind_under_p10': wind_under['p10'],
                'wind_under_p90': wind_under['p90'],
                'flood_under': flood_under['mean'],
                'flood_under_p10': flood_under['p10'],
                'flood_under_p90': flood_under['p90'],
                'total_loss': (insured_wind['mean'] + citizens['mean'] + insured_flood['mean'] + 
                              wind_under['mean'] + flood_under['mean']),
                'total_loss_p10': (insured_wind['p10'] + citizens['p10'] + insured_flood['p10'] + 
                                  wind_under['p10'] + flood_under['p10']),
                'total_loss_p90': (insured_wind['p90'] + citizens['p90'] + insured_flood['p90'] + 
                                  wind_under['p90'] + flood_under['p90']),
            }
            
            # Extract institutional stress components
            fhcf = get_scaled_values('fhcf_shortfall_usd')
            figa = get_scaled_values('figa_residual_deficit_usd')
            cit_def = get_scaled_values('citizens_residual_deficit_usd')
            nfip = get_scaled_values('nfip_borrowed_usd')
            
            institutional_data[label] = {
                'fhcf_shortfall': fhcf['mean'],
                'fhcf_shortfall_p10': fhcf['p10'],
                'fhcf_shortfall_p90': fhcf['p90'],
                'figa_residual': figa['mean'],
                'figa_residual_p10': figa['p10'],
                'figa_residual_p90': figa['p90'],
                'citizens_deficit': cit_def['mean'],
                'citizens_deficit_p10': cit_def['p10'],
                'citizens_deficit_p90': cit_def['p90'],
                'nfip_borrowed': nfip['mean'],
                'nfip_borrowed_p10': nfip['p10'],
                'nfip_borrowed_p90': nfip['p90'],
            }
else:
    print("⚠️  Climate data (df_climate) not found. Run Section 5 first to load climate scenarios.")

# ===== POLICY SCENARIOS (if available) =====
if 'policy_scenarios' in dir() and len(policy_scenarios) > 0:
    for scenario_name, df_scenario in policy_scenarios.items():
        if scenario_name == 'Baseline':
            continue  # Skip baseline as we already have it
        
        df_valid_policy = df_scenario[df_scenario['scenario'] != 'error'].copy()
        
        # Calculate losses with uncertainties
        policy_losses = {
            'insured_wind': df_valid_policy['wind_insured_private_usd'] / 1e9,
            'citizens': df_valid_policy['wind_insured_citizens_usd'] / 1e9,
            'insured_flood': df_valid_policy['flood_insured_capped_usd'] / 1e9,
            'wind_under': (df_valid_policy['wind_total_usd'] - 
                           df_valid_policy['wind_insured_private_usd'] - 
                           df_valid_policy['wind_insured_citizens_usd']) / 1e9,
            'flood_under': (df_valid_policy['water_total_usd'] - 
                            df_valid_policy['flood_insured_capped_usd']) / 1e9
        }
        
        loss_data[scenario_name] = {
            'insured_wind': policy_losses['insured_wind'].mean(),
            'insured_wind_p10': policy_losses['insured_wind'].quantile(0.10),
            'insured_wind_p90': policy_losses['insured_wind'].quantile(0.90),
            'citizens': policy_losses['citizens'].mean(),
            'citizens_p10': policy_losses['citizens'].quantile(0.10),
            'citizens_p90': policy_losses['citizens'].quantile(0.90),
            'insured_flood': policy_losses['insured_flood'].mean(),
            'insured_flood_p10': policy_losses['insured_flood'].quantile(0.10),
            'insured_flood_p90': policy_losses['insured_flood'].quantile(0.90),
            'wind_under': policy_losses['wind_under'].mean(),
            'wind_under_p10': policy_losses['wind_under'].quantile(0.10),
            'wind_under_p90': policy_losses['wind_under'].quantile(0.90),
            'flood_under': policy_losses['flood_under'].mean(),
            'flood_under_p10': policy_losses['flood_under'].quantile(0.10),
            'flood_under_p90': policy_losses['flood_under'].quantile(0.90),
        }
        
        # Add total losses
        total_loss_pol = sum([policy_losses[k].mean() for k in ['insured_wind', 'citizens', 'insured_flood', 'wind_under', 'flood_under']])
        total_loss_pol_p10 = sum([policy_losses[k].quantile(0.10) for k in ['insured_wind', 'citizens', 'insured_flood', 'wind_under', 'flood_under']])
        total_loss_pol_p90 = sum([policy_losses[k].quantile(0.90) for k in ['insured_wind', 'citizens', 'insured_flood', 'wind_under', 'flood_under']])
        
        loss_data[scenario_name]['total_loss'] = total_loss_pol
        loss_data[scenario_name]['total_loss_p10'] = total_loss_pol_p10
        loss_data[scenario_name]['total_loss_p90'] = total_loss_pol_p90
        
        # Calculate institutional data with uncertainties
        policy_inst = {
            'fhcf_shortfall': df_valid_policy['fhcf_shortfall_usd'] / 1e9,
            'figa_residual': df_valid_policy['figa_residual_deficit_usd'] / 1e9,
            'citizens_deficit': df_valid_policy['citizens_residual_deficit_usd'] / 1e9,
            'nfip_borrowed': df_valid_policy['nfip_borrowed_usd'] / 1e9
        }
        
        institutional_data[scenario_name] = {
            'fhcf_shortfall': policy_inst['fhcf_shortfall'].mean(),
            'fhcf_shortfall_p10': policy_inst['fhcf_shortfall'].quantile(0.10),
            'fhcf_shortfall_p90': policy_inst['fhcf_shortfall'].quantile(0.90),
            'figa_residual': policy_inst['figa_residual'].mean(),
            'figa_residual_p10': policy_inst['figa_residual'].quantile(0.10),
            'figa_residual_p90': policy_inst['figa_residual'].quantile(0.90),
            'citizens_deficit': policy_inst['citizens_deficit'].mean(),
            'citizens_deficit_p10': policy_inst['citizens_deficit'].quantile(0.10),
            'citizens_deficit_p90': policy_inst['citizens_deficit'].quantile(0.90),
            'nfip_borrowed': policy_inst['nfip_borrowed'].mean(),
            'nfip_borrowed_p10': policy_inst['nfip_borrowed'].quantile(0.10),
            'nfip_borrowed_p90': policy_inst['nfip_borrowed'].quantile(0.90),
        }
else:
    print("⚠️  Policy scenarios not found. Only baseline data available.")

# ===== CONVERT TO DATAFRAMES AND SAVE TO CSV =====
import pandas as pd
from pathlib import Path

# Convert loss_data to DataFrame
loss_df = pd.DataFrame.from_dict(loss_data, orient='index')
loss_df.index.name = 'Scenario'
loss_df = loss_df.reset_index()

# Convert institutional_data to DataFrame
institutional_df = pd.DataFrame.from_dict(institutional_data, orient='index')
institutional_df.index.name = 'Scenario'
institutional_df = institutional_df.reset_index()

# Save to CSV
tables_dir = Path('../results/tables')
tables_dir.mkdir(parents=True, exist_ok=True)

loss_df.to_csv(tables_dir / 'probabilistic_loss_data.csv', index=False)
institutional_df.to_csv(tables_dir / 'probabilistic_institutional_data.csv', index=False)

print(f"✓ Saved loss data to: {tables_dir / 'probabilistic_loss_data.csv'}")
print(f"✓ Saved institutional data to: {tables_dir / 'probabilistic_institutional_data.csv'}")

# Display extracted data
print("\n" + "="*120)
print("SCENARIO DATA EXTRACTED (Annual Averages in Billion USD)")
print("="*120)
print(f"\n{'Scenario':<20} {'Ins.Wind':<15} {'Citizens':<15} {'Ins.Flood':<15} {'Wind U/U':<15} {'Flood U/U':<15} {'Total':<15}")
print("-" * 120)
for scenario, data in loss_data.items():
    scenario_short = scenario.replace('\n', ' ')[:18]
    print(f"{scenario_short:<20} "
          f"${data['insured_wind']:>6.1f} ({data['insured_wind_p10']:>5.1f}-{data['insured_wind_p90']:>5.1f})  "
          f"${data['citizens']:>6.1f} ({data['citizens_p10']:>5.1f}-{data['citizens_p90']:>5.1f})  "
          f"${data['insured_flood']:>6.1f} ({data['insured_flood_p10']:>5.1f}-{data['insured_flood_p90']:>5.1f})  "
          f"${data['wind_under']:>6.1f} ({data['wind_under_p10']:>5.1f}-{data['wind_under_p90']:>5.1f})  "
          f"${data['flood_under']:>6.1f} ({data['flood_under_p10']:>5.1f}-{data['flood_under_p90']:>5.1f})  "
          f"${data['total_loss']:>6.1f} ({data['total_loss_p10']:>5.1f}-{data['total_loss_p90']:>5.1f})")

print("\n" + "="*120)
print("INSTITUTIONAL STRESS DATA (Annual Averages in Billion USD)")
print("="*120)
print(f"\n{'Scenario':<20} {'FHCF Shortfall':<30} {'FIGA Residual':<30} {'Citizens Deficit':<30} {'NFIP Borrowed':<30}")
print("-" * 120)
for scenario, data in institutional_data.items():
    scenario_short = scenario.replace('\n', ' ')[:18]
    print(f"{scenario_short:<20} "
          f"${data['fhcf_shortfall']:>6.2f} ({data['fhcf_shortfall_p10']:>6.2f}-{data['fhcf_shortfall_p90']:>6.2f})  "
          f"${data['figa_residual']:>6.2f} ({data['figa_residual_p10']:>6.2f}-{data['figa_residual_p90']:>6.2f})  "
          f"${data['citizens_deficit']:>6.2f} ({data['citizens_deficit_p10']:>6.2f}-{data['citizens_deficit_p90']:>6.2f})  "
          f"${data['nfip_borrowed']:>6.2f} ({data['nfip_borrowed_p10']:>6.2f}-{data['nfip_borrowed_p90']:>6.2f})")

print(f"\n✓ Prepared data for {len(loss_data)} scenarios")
print(f"   Loss data scenarios: {list(loss_data.keys())}")
print(f"   Institutional data scenarios: {list(institutional_data.keys())}")


## 7. Return Period Analysis

Compute return period statistics (RP10, RP25, RP50, RP100, RP250, RP500, RP1000) for losses and public burden metrics, then plot exceedance probability curves.

In [ ]:
# Function to compute return period values
def compute_return_periods(data, return_periods=[10, 25, 50, 100, 250, 500, 1000]):
    """
    Compute return period values from annual data.
    
    Parameters:
    -----------
    data : array-like
        Annual values (e.g., losses, deficits)
    return_periods : list
        Return periods to compute (in years)
    
    Returns:
    --------
    dict : Dictionary mapping return period to exceedance value
    """
    rp_values = {}
    for rp in return_periods:
        # Return period corresponds to 1/RP exceedance probability
        # e.g., RP100 = 1% annual exceedance = 99th percentile
        exceedance_prob = 1.0 / rp
        quantile = 1.0 - exceedance_prob
        rp_values[f'RP{rp}'] = np.quantile(data, quantile)
    return rp_values

# Test the function
print("Testing return period calculation on total damage:")
rp_test = compute_return_periods(df_valid['total_damage_usd'].values)
for rp_name, value in rp_test.items():
    print(f"  {rp_name}: ${value/1e9:.2f}B")

### 7.1 Compute Return Period Summary Table

Calculate annual expected values and return period statistics for all loss components and institutional metrics.

In [ ]:
# Define metrics to analyze
loss_metrics = {
    'Wind Insured (Private)': 'wind_insured_private_usd',
    'Wind Insured (Citizens)': 'wind_insured_citizens_usd',
    'Flood Insured (NFIP)': 'flood_insured_capped_usd',
    'Wind Un/Underinsured': None,  # Computed
    'Flood Un/Underinsured': None,  # Computed
}

institutional_metrics = {
    'FHCF Shortfall': 'fhcf_shortfall_usd',
    'FIGA Residual': 'figa_residual_deficit_usd',
    'Citizens Deficit': 'citizens_residual_deficit_usd',
    'NFIP Borrowed': 'nfip_borrowed_usd',
}

# Compute derived metrics
df_valid['wind_underinsured'] = (df_valid['wind_total_usd'] - 
                                   df_valid['wind_insured_private_usd'] - 
                                   df_valid['wind_insured_citizens_usd'])
df_valid['flood_underinsured'] = (df_valid['water_total_usd'] - 
                                    df_valid['flood_insured_capped_usd'])

loss_metrics['Wind Un/Underinsured'] = 'wind_underinsured'
loss_metrics['Flood Un/Underinsured'] = 'flood_underinsured'

# Compute summary statistics for all metrics
return_periods = [10, 25, 50, 100, 250, 500, 1000]
summary_data = []

print("=" * 120)
print("LOSS AND INSTITUTIONAL METRICS: ANNUAL EXPECTED VALUE AND RETURN PERIODS (Billion USD)")
print("=" * 120)

# Process loss metrics
for metric_name, column_name in loss_metrics.items():
    data = df_valid[column_name].values
    
    # Annual expected value
    annual_exp = data.mean() / 1e9
    
    # Return periods
    rp_values = compute_return_periods(data, return_periods)
    
    row = {'Metric': metric_name, 'Category': 'Loss', 'Annual EV': annual_exp}
    for rp_name, value in rp_values.items():
        row[rp_name] = value / 1e9
    
    summary_data.append(row)

# Add total losses row
df_valid['total_losses'] = (df_valid['wind_insured_private_usd'] + 
                            df_valid['wind_insured_citizens_usd'] + 
                            df_valid['flood_insured_capped_usd'] + 
                            df_valid['wind_underinsured'] + 
                            df_valid['flood_underinsured'])

total_data = df_valid['total_losses'].values
total_annual_exp = total_data.mean() / 1e9
total_rp_values = compute_return_periods(total_data, return_periods)

total_row = {'Metric': 'Total Losses', 'Category': 'Loss', 'Annual EV': total_annual_exp}
for rp_name, value in total_rp_values.items():
    total_row[rp_name] = value / 1e9

summary_data.append(total_row)

# Process institutional metrics
for metric_name, column_name in institutional_metrics.items():
    data = df_valid[column_name].values
    
    # Annual expected value
    annual_exp = data.mean() / 1e9
    
    # Return periods
    rp_values = compute_return_periods(data, return_periods)
    
    row = {'Metric': metric_name, 'Category': 'Institutional', 'Annual EV': annual_exp}
    for rp_name, value in rp_values.items():
        row[rp_name] = value / 1e9
    
    summary_data.append(row)

# Create summary DataFrame
df_summary = pd.DataFrame(summary_data)

# Save to CSV
df_summary.to_csv(tables_dir / 'baseline_metrics_return_periods.csv', index=False)
print(f"✓ Saved to: {tables_dir / 'baseline_metrics_return_periods.csv'}\n")

# Display the table
print(f"\n{'Metric':<30} {'Annual EV':>10} {'RP10':>10} {'RP25':>10} {'RP50':>10} {'RP100':>10} {'RP250':>10} {'RP500':>10} {'RP1000':>10}")
print("-" * 120)

for category in ['Loss', 'Institutional']:
    category_data = df_summary[df_summary['Category'] == category]
    print(f"\n{category.upper()}")
    print("-" * 120)
    
    for _, row in category_data.iterrows():
        print(f"{row['Metric']:<30} ${row['Annual EV']:>9.2f}B ${row['RP10']:>9.2f}B ${row['RP25']:>9.2f}B "
              f"${row['RP50']:>9.2f}B ${row['RP100']:>9.2f}B ${row['RP250']:>9.2f}B ${row['RP500']:>9.2f}B ${row['RP1000']:>9.2f}B")

print("\n" + "=" * 120)


## 8. Event-Based Return Period Analysis

Analyze individual event impacts to understand the return period of single hurricane events. This differs from annual return periods which aggregate all events in a year.

### 8.1 Load Per-Event Impact Data

Load the county-level per-event impacts from the yearset generation.

In [ ]:
# Load all events from Emanuel catalog (includes zero-damage events)
all_events_path = '../fl_risk_model/data/all_events.csv'
df_all_events = pd.read_csv(all_events_path)

print("=" * 80)
print("ALL EVENTS DATA (Emanuel ERA5 Catalog)")
print("=" * 80)
print(f"\nTotal events in catalog: {len(df_all_events):,}")

print("\nColumns available:")
print(df_all_events.columns.tolist())

print("\nSample data:")
print(df_all_events.head(10))

print("\nDamage statistics (Billion USD):")
print(f"  Events with damage >$0: {(df_all_events['total_damage_usd'] > 0).sum():,}")
print(f"  Events with zero damage: {(df_all_events['total_damage_usd'] == 0).sum():,}")
print(f"  Mean damage (all events): ${df_all_events['total_damage_usd'].mean() / 1e9:.3f}B")
print(f"  Max damage: ${df_all_events['total_damage_usd'].max() / 1e9:.2f}B")
print(f"  99th percentile: ${df_all_events['total_damage_usd'].quantile(0.99) / 1e9:.2f}B")

### 8.2 Prepare Event Data for Return Period Analysis

Filter to damaging events and prepare data for exceedance curve calculation.

In [ ]:
# Use all_events.csv data for event-based return period curve
# This includes ALL events from the Emanuel catalog (8,800 total)

# Total events in catalog
N_TOTAL_CATALOG = len(df_all_events)

# Get event damages and IDs
event_damages = df_all_events['total_damage_usd'].values
event_ids = df_all_events['event_id'].values

print("=" * 80)
print("EVENT DATA FOR RETURN PERIOD CURVE")
print("=" * 80)
print(f"\nTotal events in catalog: {N_TOTAL_CATALOG:,}")
print(f"Events with damage >$0: {(event_damages > 0).sum():,}")
print(f"Events with zero damage: {(event_damages == 0).sum():,}")

print("\nDamage statistics (Billion USD):")
print(f"  Mean: ${event_damages.mean() / 1e9:.3f}B")
print(f"  Median: ${np.median(event_damages) / 1e9:.3f}B")
print(f"  90th percentile: ${np.quantile(event_damages, 0.9) / 1e9:.2f}B")
print(f"  99th percentile: ${np.quantile(event_damages, 0.99) / 1e9:.2f}B")
print(f"  Max: ${event_damages.max() / 1e9:.2f}B")

print("\nEvents with damage > $1B:")
major_events = event_damages > 1e9
print(f"  Count: {major_events.sum():,} ({major_events.sum()/N_TOTAL_CATALOG*100:.1f}%)")

print("\nEvents with damage > $10B:")
extreme_events = event_damages > 10e9
print(f"  Count: {extreme_events.sum():,} ({extreme_events.sum()/N_TOTAL_CATALOG*100:.2f}%)")

print("\n" + "=" * 80)

### 8.3 Compute Event-Based Return Periods

Calculate return periods for individual events, accounting for event frequency.

In [ ]:
# For event-based return periods, we need to account for:
# 1. The annual frequency (λ) from the Emanuel simulations
# 2. The TOTAL number of events in the catalog (all_events.csv)
# 3. Each event's individual frequency = λ / N_total

# ERA5 Emanuel parameters (from .mat file and hazard catalog)
LAMBDA_ERA5 = 1.8  # Mean annual frequency (events/year)
N_YEARS_ERA5 = 44  # Years in ERA5 dataset

# Verify catalog size matches expected
print("=" * 80)
print("EVENT FREQUENCY ANALYSIS")
print("=" * 80)
print(f"Emanuel ERA5 parameters:")
print(f"  Mean annual frequency (λ): {LAMBDA_ERA5:.3f} events/year")
print(f"  Years in dataset: {N_YEARS_ERA5}")
print(f"\nEvent catalog (from all_events.csv):")
print(f"  Total events: {N_TOTAL_CATALOG:,}")
print(f"  Events with damage >$0: {(event_damages > 0).sum():,} ({(event_damages > 0).sum()/N_TOTAL_CATALOG*100:.1f}%)")
print(f"  Events with zero damage: {(event_damages == 0).sum():,} ({(event_damages == 0).sum()/N_TOTAL_CATALOG*100:.1f}%)")

# Each event in the catalog has an individual frequency
freq_per_event = LAMBDA_ERA5 / N_TOTAL_CATALOG

print(f"\nFrequency calculation:")
print(f"  Frequency per event: {freq_per_event:.6f} events/year")
print(f"  Return period per event: {1/freq_per_event:.0f} years")
print("=" * 80)

# Compute event exceedance curve using CLIMADA methodology
# Reference: climada.engine.impact.Impact.calc_freq_curve()

# Assign equal frequency to all events
event_frequencies = np.full(len(event_damages), freq_per_event)

# CLIMADA approach:
# 1. Sort descendingly by impact
sort_idxs = np.argsort(event_damages)[::-1]

# 2. Calculate cumulative exceedance frequency (events/year exceeding each threshold)
exceed_freq = np.cumsum(event_frequencies[sort_idxs])

# 3. Compute return periods and reverse arrays to ascending order
event_return_period = (1 / exceed_freq)[::-1]
event_damages_sorted = event_damages[sort_idxs][::-1]

print("\nEvent-based return period statistics (CLIMADA method):")
print(f"  Shortest RP: {event_return_period.min():.2f} years")
print(f"  Longest RP: {event_return_period.max():.0f} years")
print(f"  Mean exceedance frequency: {exceed_freq.mean():.6f} events/year")
print(f"  Max exceedance frequency: {exceed_freq.max():.6f} events/year")

# Find key event return periods
print("\nDamage at standard return periods:")
for rp_target in [10, 50, 100, 250, 500, 1000]:
    # Interpolate to find damage at exact RP
    damage_at_rp = np.interp(rp_target, event_return_period, event_damages_sorted)
    print(f"  RP{rp_target}: ${damage_at_rp/1e9:.2f}B")

### 8.4 Plot Event-Based and Event-Set Based Return Period Curves

Visualize both return period curves:
- **Event-Based**: Individual hurricane event impacts from the catalog
- **Event-Set Based (Seasonal)**: Annual aggregated damages from the 10,000 year MC simulation


In [ ]:
# Create event-based return period curve
fig, ax = plt.subplots(1, 1, figsize=(4, 3))

# Compute event-set based (seasonal) return period curve from MC simulation
# Use the 10,000 year seasonal damages from df_valid
seasonal_damages = df_valid['total_damage_usd'].values
seasonal_damages_sorted = np.sort(seasonal_damages)[::-1]  # Sort descending
n_years = len(seasonal_damages_sorted)
seasonal_return_periods = n_years / np.arange(1, n_years + 1)  # RP = N / rank

# Plot event-set based (seasonal) curve
ax.plot(seasonal_return_periods, seasonal_damages_sorted / 1e9,
        linewidth=2.5, label='Year set-based', color='k', 
        alpha=0.85)

# Plot event-based curve (RP on x-axis, Damage on y-axis)
ax.plot(event_return_period, event_damages_sorted / 1e9,
        linewidth=2.5, label='Event-based', color='#E74C3C', 
        alpha=0.85, linestyle='--')

ax.set_xscale('log')
ax.set_yscale('log')

ax.set_ylim(bottom=1e-3, top=2e3)  # Set y-axis minimum to 1 million USD (1e-3 billion)
ax.set_xlim(left=1, right=5000)  
ax.set_xlabel('Return Period (Years)', fontsize=11, fontweight='normal')
ax.set_ylabel('Total Loss (Billion USD)', fontsize=11, fontweight='normal')

# Set explicit tick locations and enable tick marks
ax.set_xticks([1, 10, 100, 1000])
ax.set_yticks([1e-3, 1e-2, 1e-1, 1, 10, 100, 1000])

# Show only bottom and left spines
ax.spines['bottom'].set_visible(True)
ax.spines['left'].set_visible(True)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

ax.spines['bottom'].set_color('black')
ax.spines['left'].set_color('black')
ax.spines['bottom'].set_linewidth(1)
ax.spines['left'].set_linewidth(1)

# Enable tick marks (seaborn whitegrid hides them by default)
ax.tick_params(axis='both', which='major', bottom=True, left=True, top=False, right=False,
               direction='out', length=4, width=1.0, color='k', labelsize=10)

ax.grid(True, alpha=0.3, which='both', linestyle='-', linewidth=0.5)

# Add legend
ax.legend(fontsize=8, frameon=True, loc='lower right')

plt.tight_layout()

# Save figure
out_file = Path('../results/figures/fig_loss_return_period.png')
plt.savefig(out_file, dpi=300, bbox_inches='tight')
plt.show()



### Return Period Curves with Climate Scenarios

Comparing baseline (ERA5) return period curves with future climate scenarios to visualize how climate change shifts the risk landscape.

In [ ]:
# ============================================================================
# LOAD AND POOL GCM SCENARIO ANNUAL DATA FOR RP CURVE ANALYSIS
# ============================================================================
# Strategy: Pool all 5 GCMs for each SSP/timeframe to create robust RP curves
# This gives ~50,000 years of data per scenario (5 GCMs × 10,000 years each)
# ============================================================================

import pandas as pd
from pathlib import Path

# Define all available GCMs
gcm_list = ['canesm', 'cnrm6', 'ecearth6', 'ipsl6', 'miroc6']

# Define scenario configurations
# Format: (label, scenario_suffix, time_category)
scenario_configs = [
    ('2050 SSP2-4.5', 'ssp245cal', 'near'),
    ('2050 SSP5-8.5', 'ssp585cal', 'near'),
    ('2100 SSP2-4.5', 'ssp245_2cal', 'mid'),
    ('2100 SSP5-8.5', 'ssp585_2cal', 'mid')
]

# Pool data across all GCMs for each scenario
scenario_annual_data = {}
mc_root = Path('../results/mc_runs')

print("="*80)
print("LOADING AND POOLING MULTI-GCM SCENARIO DATA")
print("="*80)

for label, scenario_suffix, time_cat in scenario_configs:
    pooled_data = []
    gcm_count = 0
    
    print(f"\n{label}:")
    
    for gcm in gcm_list:
        # Find the most recent run directory for this GCM+scenario
        pattern = f"emanuel_{gcm}_{scenario_suffix}_baseline_*"
        matching_dirs = list(mc_root.glob(pattern))
        
        if not matching_dirs:
            print(f"  ⚠️  {gcm}: No data found")
            continue
        
        # Use the most recent directory
        run_dir = sorted(matching_dirs)[-1]
        iterations_file = run_dir / 'iterations.csv'
        
        if iterations_file.exists():
            # Load the annual data
            df_gcm = pd.read_csv(iterations_file)
            
            # Filter out error scenarios
            df_gcm_valid = df_gcm[df_gcm['scenario'] != 'error'].copy()
            
            # Add GCM identifier
            df_gcm_valid['gcm'] = gcm
            
            pooled_data.append(df_gcm_valid)
            gcm_count += 1
            
            print(f"  ✓ {gcm:12s}: {len(df_gcm_valid):,} years")
        else:
            print(f"  ⚠️  {gcm}: iterations.csv not found")
    
    if pooled_data:
        # Concatenate all GCMs for this scenario
        df_pooled = pd.concat(pooled_data, ignore_index=True)
        scenario_annual_data[label] = df_pooled
        
        print(f"  → POOLED: {len(df_pooled):,} years from {gcm_count} GCMs")
    else:
        print(f"  ✗ No data available for {label}")

print(f"\n{'='*80}")
print(f"✓ Pooled data for {len(scenario_annual_data)}/4 climate scenarios")
print(f"  Scenarios: {list(scenario_annual_data.keys())}")
print(f"  Total years per scenario: ~{len(next(iter(scenario_annual_data.values()))):,}")

In [ ]:
# ============================================================================
# VISUALIZE CLIMATE SCENARIO RP CURVES WITH BASELINE OVERLAY
# ============================================================================

# Create 2x3 subplot grid (6 metrics)
fig, axes = plt.subplots(2, 3, figsize=(12, 6))
axes = axes.flatten()

# Color scheme
color_baseline  = "#6B7280"  # neutral gray
color_2050_245  = '#FBBF24'  # lighter amber  
color_2050_585  = '#D97706'  # darker amber
color_2100_245  = '#FCA5A5'  # lighter red
color_2100_585  = '#B91C1C'  # darker red
#color_threshold = '#DC2626'  # Red for threshold
color_threshold = 'k'  # Red for threshold

color_map = {
    '2050 SSP2-4.5': color_2050_245,
    '2050 SSP5-8.5': color_2050_585,
    '2100 SSP2-4.5': color_2100_245,
    '2100 SSP5-8.5': color_2100_585
}

# Metrics to analyze
metrics_to_plot = [
    ('defaults_post', 'Number of Company Defaults', 10, 'count'),
    ('largest_entity_deficit_usd', 'Single Largest Company Deficit (bn USD)', 1.0, 'billions'),
    ('fhcf_shortfall_usd', 'FHCF Shortfall (bn USD)', 17.0, 'billions'),
    ('figa_residual_deficit_usd', 'FIGA Residual (bn USD)', 4.0, 'billions'),
    ('citizens_residual_deficit_usd', 'Citizens Deficit (bn USD)', 11.7, 'billions'),
    ('nfip_borrowed_usd', 'NFIP Amount Borrowed (bn USD)', 3.0, 'billions')
]

# Plot each metric
for idx, (metric, ylabel, threshold_val, unit) in enumerate(metrics_to_plot):
    ax = axes[idx]
    
    # 1. Plot baseline RP curve
    values_base, rp_base = calculate_return_period_curve(df_valid[metric])
    
    # Convert to billions if needed
    if unit == 'billions':
        values_base = values_base / 1e9
    
    ax.plot(rp_base, values_base, linewidth=2.5, color=color_baseline, 
            alpha=0.85, label='ERA5 Baseline', zorder=5)
    
    # 2. Plot climate scenario RP curves
    for scenario_label, df_scenario in scenario_annual_data.items():
        values_climate, rp_climate = calculate_return_period_curve(df_scenario[metric])
        
        # Convert to billions if needed
        if unit == 'billions':
            values_climate = values_climate / 1e9
        
        ax.plot(rp_climate, values_climate, linewidth=2, 
                color=color_map[scenario_label], alpha=0.8,
                label=scenario_label, zorder=4)
    
    # 3. Add threshold marker
    if values_base.min() <= threshold_val <= values_base.max():
        threshold_rp = np.interp(threshold_val, values_base[::-1], rp_base[::-1])
        
        # Add horizontal line at threshold
        ax.axhline(threshold_val, color=color_threshold, linestyle='--', 
                  linewidth=1,  zorder=1)
        
        # # Add vertical line at return period
        # ax.axvline(threshold_rp, color=color_threshold, linestyle='--', 
        #           linewidth=1,  zorder=1)
    
    # Set log-log scale
    ax.set_xscale('log')
    ax.set_yscale('log')
    
    # Set axis limits
    ax.set_xlim(left=1, right=max(rp_base) * 1.2)
    
    # Determine appropriate y-axis limits based on data and metric type
    if unit == 'count':
        y_min = 1  # Minimum 1 default (can't have 0.5 defaults on log scale)
    elif unit == 'billions':
        y_min = 0.01  # Minimum $0.01B = $10M
    else:
        y_min = max(values_base.min() * 0.5, 1e-3)
    
    y_max = values_base.max() * 2
    ax.set_ylim(bottom=y_min, top=y_max)
    
    # Add subplot label (a, b, c, d, e, f)
    subplot_labels = ['a', 'b', 'c', 'd', 'e', 'f']
    ax.text(-0.1, 1.05, subplot_labels[idx], transform=ax.transAxes,
            fontsize=12, fontweight='bold', va='top', ha='left')
    
    # Labels - show x-axis label on bottom row
    if idx >= 3:  # Bottom row (indices 3, 4, 5)
        ax.set_xlabel('Return Period (Years)', fontsize=9, fontweight='normal')
    else:
        ax.set_xlabel('')
    
    ax.set_ylabel(ylabel, fontsize=9, fontweight='normal')
    
    # Show only bottom and left spines
    ax.spines['bottom'].set_visible(True)
    ax.spines['left'].set_visible(True)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    ax.spines['bottom'].set_color('black')
    ax.spines['left'].set_color('black')
    ax.spines['bottom'].set_linewidth(1)
    ax.spines['left'].set_linewidth(1)
    
    # Enable tick marks
    ax.tick_params(axis='both', which='major', bottom=True, left=True, top=False, right=False,
                   direction='out', length=4, width=1.0, color='k', labelsize=8)
    
    # Grid
    ax.grid(True, alpha=0.3, which='both', linestyle='-', linewidth=0.5)

# Create legend on the right side of the figure
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

legend_handles = [
    Patch(facecolor=color_baseline, edgecolor='black', label='ERA5 Baseline'),
    Patch(facecolor=color_2050_245, edgecolor='black', label='2050 SSP2-4.5'),
    Patch(facecolor=color_2050_585, edgecolor='black', label='2050 SSP5-8.5'),
    Patch(facecolor=color_2100_245, edgecolor='black', label='2100 SSP2-4.5'),
    Patch(facecolor=color_2100_585, edgecolor='black', label='2100 SSP5-8.5'),
    Line2D([0], [0], color='black', linestyle='--', linewidth=1.0, label='Threshold'),
]

fig.legend(handles=legend_handles, loc='center left', bbox_to_anchor=(1.02, 0.5),
           fontsize=9, frameon=True, fancybox=False, edgecolor='black', 
           framealpha=0.95, title='Scenario', title_fontsize=10)

plt.tight_layout()

# Save figure
out_file = Path('../results/figures/fig_return_period_climate_overlay.png')
plt.savefig(out_file, dpi=300, bbox_inches='tight')
plt.show()

print(f"\n✓ Climate RP curves saved to: {out_file}")
